# 🚀 CausalCrisis V3 Enhanced — Experiment Notebook

**Multimodal Causal Classification for Crisis Events**

---

## Kiến trúc V3 — 5 Key Improvements

| # | Cải tiến | Nguồn | Impact |
|---|---------|-------|--------|
| 1 | Hybrid ICA-Adversarial Disentanglement | CCA (Jiang 2025) | +1-2% F1 |
| 2 | Supervised Contrastive Loss | SupCon (Khosla 2020) | +0.5-1% F1 |
| 3 | Bilinear Fusion option | Simplicity Test | Ablation |
| 4 | Adaptive Loss Weighting | Kendall 2018 | -50% tuning |
| 5 | 2-Phase Training + Warm Restarts | Loshchilov 2017 | Simpler |

## Pipeline

```
CLIP ViT-L/14 (frozen) → HybridDisentangler → CrossAttention/Bilinear → Classifier → BA
```

**Target:** >90% F1 on CrisisMMD Task 1 (surpass CrisisSpot 90.9%)

**Runtime:** GPU A100 recommended

---
## 📦 Cell 1: Setup & Installation

In [ ]:
# ============================================================================
# 📦 CELL 1: Setup & Installation
# ============================================================================
print("=" * 60)
print("🚀 CausalCrisis V3 Enhanced — Setup")
print("=" * 60)

!pip install -q open_clip_torch scikit-learn pandas matplotlib seaborn tqdm PyYAML

import torch
import numpy as np
import os
import sys
import time

print(f"\n✅ PyTorch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Training will be very slow.")

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted")
except ImportError:
    IN_COLAB = False
    print("ℹ️ Not in Colab — running locally")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n🖥️ Using device: {DEVICE}")

---
## 📂 Cell 2: Clone Repository & Import Modules

In [ ]:
# ============================================================================
# 📂 CELL 2: Clone Repository & Import Modules
# ============================================================================
print("=" * 60)
print("📂 Setting up project")
print("=" * 60)

# === CẤU HÌNH ĐƯỜNG DẪN — Thay đổi theo setup của bạn ===
REPO_URL = "https://github.com/Thanh-000/CrisisSummarization-CausalGNN.git"
BRANCH = "v3-causalcrisis-enhanced"
PROJECT_DIR = "/content/CausalCrisisV3"

# Đường dẫn dataset CrisisMMD v2.0
# Option 1: Upload trực tiếp lên Colab
DATASET_DIR = "/content/CrisisMMD_v2.0"
# Option 2: Từ Google Drive (uncomment nếu dùng)
# DATASET_DIR = "/content/drive/MyDrive/datasets/CrisisMMD_v2.0"

# Cache directory cho CLIP features (tránh extract lại)
CACHE_DIR = "/content/cached_features"

# Clone repo nếu chưa có
if not os.path.exists(os.path.join(PROJECT_DIR, "src")):
    print(f"📥 Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print("✅ Repository already exists")
    # Pull latest changes
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

# Thêm project vào Python path
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("\n📦 Importing CausalCrisis V3 modules...")

# Import tất cả modules
from src.config import get_config, CausalCrisisConfig
from src.models import CausalCrisisV3, CLIPMLPBaseline
from src.losses import CausalCrisisLoss, FocalLoss
from src.data import (
    load_crisismmd_annotations,
    extract_and_cache_clip_features,
    create_stratified_splits,
    create_dataloaders,
    compute_class_weights,
)
from src.trainer import CausalCrisisTrainer, BaselineTrainer
from src.evaluate import (
    compute_metrics,
    paired_bootstrap_test,
    plot_training_curves,
    plot_tsne_causal_features,
)

print("✅ All modules imported successfully!")

# Hiển thị cấu trúc src/
print("\n📁 Project structure:")
!ls -la {PROJECT_DIR}/src/*.py

---
## 📊 Cell 3: Load CrisisMMD Dataset

> **Lưu ý:** Bạn cần upload/có sẵn dataset CrisisMMD v2.0.  
> Download tại: https://crisisnlp.qcri.org/crisismmd

In [ ]:
# ============================================================================
# 📊 CELL 3: Load CrisisMMD Dataset
# ============================================================================
print("=" * 60)
print("📊 Loading CrisisMMD v2.0")
print("=" * 60)

# Kiểm tra dataset tồn tại
if not os.path.exists(DATASET_DIR):
    print(f"❌ Dataset not found at: {DATASET_DIR}")
    print("\n📋 Hướng dẫn:")
    print("   1. Download CrisisMMD v2.0 từ https://crisisnlp.qcri.org/crisismmd")
    print("   2. Upload lên Colab hoặc Google Drive")
    print(f"   3. Đảm bảo đường dẫn đúng: {DATASET_DIR}")
    print("\n   Hoặc upload trực tiếp:")
    print("   from google.colab import files")
    print("   files.upload()  # upload file zip")
    print("   !unzip CrisisMMD_v2.0.zip -d /content/")
else:
    # Load annotations
    data = load_crisismmd_annotations(DATASET_DIR, task="task1")
    
    print(f"\n✅ Dataset loaded successfully!")
    print(f"   Shape: {data.shape}")
    print(f"   Columns: {list(data.columns)}")
    print(f"\n📊 Label distribution:")
    print(data["label"].value_counts())
    
    if "event" in data.columns:
        print(f"\n🌍 Events (disasters):")
        print(data["event"].value_counts())

---
## 🔄 Cell 4: CLIP Feature Extraction & Caching

> Sử dụng CLIP **ViT-L/14** (768-dim) từ OpenAI.  
> Features được cache thành `.npy` files → chỉ extract 1 lần.

In [ ]:
# ============================================================================
# 🔄 CELL 4: CLIP Feature Extraction & Caching
# ============================================================================
print("=" * 60)
print("🔄 CLIP Feature Extraction (ViT-L/14)")
print("=" * 60)

config = get_config("task1")
print(f"   Model: {config.clip.model_name}")
print(f"   Feature dim: {config.clip.image_dim}")
print(f"   Cache dir: {CACHE_DIR}")

os.makedirs(CACHE_DIR, exist_ok=True)

# Extract features (hoặc load từ cache)
t0 = time.time()
image_features, text_features = extract_and_cache_clip_features(
    data=data,
    cache_dir=CACHE_DIR,
    model_name=config.clip.model_name,
    batch_size=64,
    device=DEVICE,
)
elapsed = time.time() - t0

# Labels & domain IDs
labels = data["label"].values
if "domain_id" in data.columns:
    domain_ids = data["domain_id"].values.astype(int)
elif "event" in data.columns:
    # Tạo domain_id từ event names
    events = data["event"].unique()
    event_to_id = {e: i for i, e in enumerate(events)}
    domain_ids = data["event"].map(event_to_id).values.astype(int)
else:
    domain_ids = np.zeros(len(labels), dtype=int)

print(f"\n✅ Feature extraction complete ({elapsed:.1f}s)")
print(f"\n📐 Feature shapes:")
print(f"   Image features: {image_features.shape}")
print(f"   Text features:  {text_features.shape}")
print(f"   Labels:  {labels.shape} → classes: {np.unique(labels)} (counts: {np.bincount(labels)})")
print(f"   Domains: {len(np.unique(domain_ids))} unique domains")

---
## 📊 Cell 5: Create Data Splits & DataLoaders

In [ ]:
# ============================================================================
# 📊 CELL 5: Create Data Splits & DataLoaders
# ============================================================================
print("=" * 60)
print("📊 Creating Stratified Splits")
print("=" * 60)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Stratified split (giữ tỷ lệ class + domain)
train_idx, val_idx, test_idx = create_stratified_splits(
    labels, domain_ids,
    test_ratio=config.eval.test_split,   # 0.15
    val_ratio=config.eval.val_split,      # 0.15
    seed=SEED,
)

# Class weights cho Focal Loss
train_labels = labels[train_idx]
class_weights = compute_class_weights(train_labels)

# DataLoaders
loaders = create_dataloaders(
    image_features, text_features, labels, domain_ids,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    batch_size=config.training.batch_size,  # 128
    num_workers=2,  # Colab stable value
)

print(f"\n✅ Data splits created:")
print(f"   Train: {len(loaders['train'].dataset):,} samples")
print(f"   Val:   {len(loaders['val'].dataset):,} samples")
print(f"   Test:  {len(loaders['test'].dataset):,} samples")
print(f"\n⚖️ Class weights: {class_weights}")
print(f"   Train label distribution: {np.bincount(train_labels)}")

---
## 🧪 Cell 6: Experiment H1 — CLIP MLP Baseline

> **Hypothesis H1:** CLIP ViT-L/14 features + simple MLP achieves F1 > 88%  
> Đây là baseline để so sánh với CausalCrisis V3

In [ ]:
# ============================================================================
# 🧪 CELL 6: EXPERIMENT H1 — CLIP MLP Baseline
# ============================================================================
print("=" * 60)
print("🧪 H1: CLIP ViT-L/14 + MLP Baseline")
print("   Prediction: F1 > 88%")
print("=" * 60)

baseline_results = []
baseline_histories = []

# Chạy với multiple seeds để đánh giá stability
BASELINE_SEEDS = [42, 123, 456]

for seed in BASELINE_SEEDS:
    print(f"\n{'─'*40}")
    print(f"🌱 Seed {seed}")
    print(f"{'─'*40}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # Model: concat(image, text) → MLP
    baseline_model = CLIPMLPBaseline(
        input_dim=config.clip.image_dim * 2,  # 768*2 = 1536
        num_classes=config.classifier.num_classes,
    )
    n_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
    print(f"   Parameters: {n_params:,}")
    
    # Optimizer + Scheduler
    optimizer = torch.optim.AdamW(
        baseline_model.parameters(),
        lr=config.training.lr,
        weight_decay=config.training.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=100, eta_min=1e-6
    )
    
    # Loss: Focal Loss with class weights
    loss_fn = FocalLoss(alpha=class_weights, gamma=config.training.focal_gamma)
    
    # Train
    trainer = BaselineTrainer(
        baseline_model, optimizer, scheduler,
        device=DEVICE, save_dir="checkpoints"
    )
    history = trainer.train(
        loaders["train"], loaders["val"],
        epochs=100, patience=15,
        loss_fn=loss_fn,
    )
    baseline_histories.append(history)
    
    # Load best model and evaluate on TEST set
    baseline_model.load_state_dict(
        torch.load("checkpoints/baseline_best.pt", map_location=DEVICE)
    )
    baseline_model.eval().to(DEVICE)
    
    all_preds, all_labels_test, all_probs = [], [], []
    with torch.no_grad():
        for batch in loaders["test"]:
            f_v = batch["image_features"].to(DEVICE)
            f_t = batch["text_features"].to(DEVICE)
            logits = baseline_model(f_v, f_t)
            all_preds.extend(logits.argmax(-1).cpu().numpy())
            all_labels_test.extend(batch["label"].numpy())
            all_probs.extend(torch.softmax(logits, -1).cpu().numpy())
    
    metrics = compute_metrics(
        np.array(all_labels_test), np.array(all_preds), np.array(all_probs)
    )
    metrics["predictions"] = np.array(all_preds)   # Lưu để so sánh sau
    metrics["labels"] = np.array(all_labels_test)
    baseline_results.append(metrics)
    
    print(f"\n   📊 Test Results:")
    print(f"      F1 (weighted) = {metrics['f1_weighted']:.4f}")
    print(f"      Accuracy      = {metrics['accuracy']:.4f}")
    print(f"      Precision     = {metrics['precision']:.4f}")
    print(f"      Recall        = {metrics['recall']:.4f}")

### 📊 H1 Summary

In [ ]:
# ============================================================================
# 📊 H1 SUMMARY
# ============================================================================
f1_baseline = [r['f1_weighted'] for r in baseline_results]
acc_baseline = [r['accuracy'] for r in baseline_results]

print("\n" + "=" * 60)
print("📊 H1 Results: CLIP ViT-L/14 + MLP Baseline")
print("=" * 60)
print(f"   Seeds:    {BASELINE_SEEDS}")
print(f"   F1:       {np.mean(f1_baseline):.4f} ± {np.std(f1_baseline):.4f}")
print(f"   Accuracy: {np.mean(acc_baseline):.4f} ± {np.std(acc_baseline):.4f}")
print(f"   Best F1:  {np.max(f1_baseline):.4f}")
print(f"   Worst F1: {np.min(f1_baseline):.4f}")

h1_supported = np.mean(f1_baseline) > 0.88
print(f"\n   H1 supported (F1 > 88%): {'✅ YES' if h1_supported else '❌ NO'}")
print("=" * 60)

# Classification report cho best run
best_idx = np.argmax(f1_baseline)
print(f"\n📋 Classification Report (best seed = {BASELINE_SEEDS[best_idx]}):")
print(baseline_results[best_idx]['classification_report'])

---
## 🧪 Cell 7: Experiment H2/H3/H5 — CausalCrisis V3 Full Model

> **Target:** F1 > 90% (surpass CrisisSpot 90.9%)  
> Bao gồm: Hybrid Disentanglement + CrossAttention + SupCon + Adaptive Weights + BA

In [ ]:
# ============================================================================
# 🧪 CELL 7: CausalCrisis V3 Full Model
# ============================================================================
print("=" * 60)
print("🧪 H2/H3/H5: CausalCrisis V3 Enhanced")
print("   Target: F1 > 90% (surpass CrisisSpot 90.9%)")
print("=" * 60)

v3_results = []
v3_histories = []

V3_SEEDS = [42, 123, 456]

for seed in V3_SEEDS:
    print(f"\n{'═'*50}")
    print(f"🌱 Seed {seed}")
    print(f"{'═'*50}")
    
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # ── Build CausalCrisis V3 model ──
    model = CausalCrisisV3(
        input_dim=config.clip.image_dim,          # 768
        causal_dim=config.disentangle.causal_dim,  # 384
        spurious_dim=config.disentangle.spurious_dim,  # 384
        num_classes=config.classifier.num_classes,  # 2
        num_domains=config.training.num_domains,    # 7
        nhead=config.fusion.nhead,                  # 8
        dropout=config.disentangle.dropout,         # 0.1
        use_ica_init=config.disentangle.use_ica_init,  # True
        fusion_type=config.fusion.fusion_type,      # "cross_attention"
        grl_lambda_max=config.training.grl_lambda_max,  # 1.0
        grl_warmup_epochs=config.training.grl_warmup_epochs,  # 10
    )
    
    print(f"📐 Model parameters: {model.get_trainable_params():,}")
    print(f"   Fusion type: {config.fusion.fusion_type}")
    print(f"   ICA init: {config.disentangle.use_ica_init}")
    print(f"   Adaptive weights: {config.training.use_adaptive_weights}")
    
    # ── Loss function ──
    loss_fn = CausalCrisisLoss(
        num_classes=config.classifier.num_classes,
        focal_gamma=config.training.focal_gamma,
        alpha_adv=config.training.alpha_adv,
        alpha_ortho=config.training.alpha_ortho,
        alpha_supcon=config.training.alpha_supcon,
        use_adaptive=config.training.use_adaptive_weights,
        class_weights=class_weights,
    )
    
    # ── Optimizer (model + loss params) ──
    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(loss_fn.parameters()),
        lr=config.training.lr,
        weight_decay=config.training.weight_decay,
    )
    
    # ── Scheduler: Cosine Annealing with Warm Restarts ──
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=config.training.T_0,      # 30
        T_mult=config.training.T_mult, # 2
        eta_min=config.training.eta_min,
    )
    
    # ── Trainer ──
    trainer = CausalCrisisTrainer(
        model=model,
        loss_fn=loss_fn,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        warmup_epochs=config.training.warmup_epochs,
        ba_start_epoch=config.training.ba_start_epoch,
        save_dir="checkpoints",
        experiment_name=f"v3_seed{seed}",
    )
    
    # ── Train ──
    history = trainer.train(
        loaders["train"], loaders["val"],
        epochs=config.training.epochs,
        patience=config.training.early_stop_patience,
    )
    v3_histories.append(history)
    
    # ── Evaluate on test (with Backdoor Adjustment) ──
    test_metrics = trainer.evaluate(loaders["test"], use_ba=True)
    v3_results.append(test_metrics)
    
    print(f"\n📊 Test Results (seed {seed}):")
    print(f"   F1 (weighted) = {test_metrics['f1']:.4f}")
    print(f"   Accuracy      = {test_metrics['accuracy']:.4f}")
    print(f"   Precision     = {test_metrics['precision']:.4f}")
    print(f"   Recall        = {test_metrics['recall']:.4f}")

### 📊 V3 Summary & Comparison

In [ ]:
# ============================================================================
# 📊 V3 SUMMARY & COMPARISON WITH BASELINE
# ============================================================================
f1_v3 = [r['f1'] for r in v3_results]
acc_v3 = [r['accuracy'] for r in v3_results]

print("\n" + "=" * 60)
print("📊 CausalCrisis V3 Enhanced — Final Results")
print("=" * 60)

print(f"\n{'Method':<30} {'F1 Mean':>10} {'F1 Std':>10} {'Best':>10}")
print(f"{'─'*60}")
print(f"{'CLIP MLP Baseline':<30} {np.mean(f1_baseline):>10.4f} {np.std(f1_baseline):>10.4f} {np.max(f1_baseline):>10.4f}")
print(f"{'CausalCrisis V3':<30} {np.mean(f1_v3):>10.4f} {np.std(f1_v3):>10.4f} {np.max(f1_v3):>10.4f}")
print(f"{'CrisisSpot (SOTA)':<30} {'90.90':>10} {'—':>10} {'—':>10}")

delta = np.mean(f1_v3) - np.mean(f1_baseline)
print(f"\n   Δ V3 vs Baseline: {delta:+.4f}")
print(f"   Δ V3 vs CrisisSpot: {np.mean(f1_v3) - 0.909:+.4f}")

beat_crisisspot = np.mean(f1_v3) > 0.909
print(f"\n   🏆 Surpass CrisisSpot (90.9%): {'✅ YES!' if beat_crisisspot else '❌ Not yet'}")
print("=" * 60)

---
## 📈 Cell 8: Training Visualization

In [ ]:
# ============================================================================
# 📈 CELL 8: Training Visualization
# ============================================================================
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print("📈 Generating training visualizations...")

# Plot V3 training curves (last seed)
if v3_histories:
    plot_training_curves(v3_histories[-1], save_path="v3_training_curves.png")

# Comparison plot: Baseline vs V3
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 comparison bar chart
methods = ['Baseline\n(CLIP MLP)', 'CausalCrisis\nV3', 'CrisisSpot\n(SOTA)']
f1_means = [np.mean(f1_baseline), np.mean(f1_v3), 0.909]
f1_stds = [np.std(f1_baseline), np.std(f1_v3), 0]
colors = ['#4ECDC4', '#FF6B6B', '#95E1D3']

bars = axes[0].bar(methods, f1_means, yerr=f1_stds, capsize=5,
                   color=colors, edgecolor='white', linewidth=2, alpha=0.85)
axes[0].set_ylabel('Weighted F1 Score', fontsize=12)
axes[0].set_title('Model Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylim(0.8, 1.0)
axes[0].axhline(y=0.909, color='gray', linestyle='--', alpha=0.5, label='CrisisSpot SOTA')
for bar, mean in zip(bars, f1_means):
    axes[0].text(bar.get_x() + bar.get_width()/2., mean + 0.005,
                f'{mean:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Val F1 over training (last seeds)
if baseline_histories and v3_histories:
    axes[1].plot(baseline_histories[-1]['val_f1'], label='Baseline', alpha=0.8, color='#4ECDC4')
    axes[1].plot(v3_histories[-1]['val_f1'], label='V3 Enhanced', alpha=0.8, color='#FF6B6B')
    axes[1].axhline(y=0.909, color='gray', linestyle='--', alpha=0.5, label='CrisisSpot')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Val F1', fontsize=12)
    axes[1].set_title('Validation F1 Over Training', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: comparison_plot.png")

---
## 🔬 Cell 9: t-SNE Visualization of Causal Features

> Causal features nên cluster theo **class**, KHÔNG theo **domain** (disaster type).  
> Điều này cho thấy model đã loại bỏ spurious domain correlations.

In [ ]:
# ============================================================================
# 🔬 CELL 9: t-SNE Visualization of Causal Features
# ============================================================================
print("🔬 t-SNE Visualization of Causal Features...")

# Load best V3 model
model.eval()
all_C_v, all_C_t = [], []
all_labels_viz, all_domains_viz = [], []

with torch.no_grad():
    for batch in loaders["test"]:
        f_v = batch["image_features"].to(DEVICE)
        f_t = batch["text_features"].to(DEVICE)
        
        output = model(f_v, f_t)
        all_C_v.append(output["C_v"].cpu().numpy())
        all_C_t.append(output["C_t"].cpu().numpy())
        all_labels_viz.extend(batch["label"].numpy())
        if "domain_id" in batch:
            all_domains_viz.extend(batch["domain_id"].numpy())

C_v_all = np.concatenate(all_C_v)
C_t_all = np.concatenate(all_C_t)
labels_viz = np.array(all_labels_viz)
domains_viz = np.array(all_domains_viz) if all_domains_viz else np.zeros(len(labels_viz))

# Plot
plot_tsne_causal_features(
    C_v_all, C_t_all, labels_viz, domains_viz,
    save_path="tsne_causal_features.png"
)
print("✅ Saved: tsne_causal_features.png")

---
## 📏 Cell 10: Statistical Significance Test

> Paired Bootstrap Test (10,000 iterations).  
> H₀: Không có sự khác biệt giữa V3 và Baseline.

In [ ]:
# ============================================================================
# 📏 CELL 10: Statistical Significance Test
# ============================================================================
print("=" * 60)
print("📏 Paired Bootstrap Significance Test")
print("=" * 60)

# Lấy predictions từ best run của mỗi model
best_baseline_idx = np.argmax(f1_baseline)
best_v3_idx = np.argmax(f1_v3)

baseline_preds = baseline_results[best_baseline_idx].get("predictions")
v3_preds = v3_results[best_v3_idx]["predictions"]
test_labels_arr = v3_results[best_v3_idx]["labels"]

if baseline_preds is not None and len(baseline_preds) == len(v3_preds):
    sig_test = paired_bootstrap_test(
        test_labels_arr, v3_preds, baseline_preds,
        n_bootstrap=10000,
    )
    
    print(f"\n   V3 F1:        {sig_test['score_a']:.4f}")
    print(f"   Baseline F1:  {sig_test['score_b']:.4f}")
    print(f"   Difference:   {sig_test['observed_diff']:+.4f}")
    print(f"   p-value:      {sig_test['p_value']:.6f}")
    print(f"   95% CI:       [{sig_test['ci_lower']:+.4f}, {sig_test['ci_upper']:+.4f}]")
    
    if sig_test['significant']:
        print(f"\n   ✅ SIGNIFICANT (p < 0.05) — V3 is significantly better!")
    else:
        print(f"\n   ❌ NOT significant (p ≥ 0.05)")
else:
    print("\n⚠️ Cannot compare — prediction arrays have different sizes")
    print("   Run both experiments with the same test split.")

---
## 💾 Cell 11: Save Results & Export

In [ ]:
# ============================================================================
# 💾 CELL 11: Save Results
# ============================================================================
import json

print("=" * 60)
print("💾 Saving experiment results")
print("=" * 60)

os.makedirs("results", exist_ok=True)

# Save summary
summary = {
    "experiment": "CausalCrisis V3 Enhanced",
    "dataset": "CrisisMMD v2.0 Task 1",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "baseline": {
        "model": "CLIP ViT-L/14 + MLP",
        "seeds": BASELINE_SEEDS,
        "f1_mean": float(np.mean(f1_baseline)),
        "f1_std": float(np.std(f1_baseline)),
        "f1_scores": [float(f) for f in f1_baseline],
    },
    "v3": {
        "model": "CausalCrisis V3 Enhanced",
        "seeds": V3_SEEDS,
        "f1_mean": float(np.mean(f1_v3)),
        "f1_std": float(np.std(f1_v3)),
        "f1_scores": [float(f) for f in f1_v3],
        "improvements": [
            "Hybrid ICA-Adversarial Disentanglement",
            "Supervised Contrastive Loss",
            "Adaptive Loss Weighting",
            "2-Phase Training + Warm Restarts",
            "Backdoor Adjustment",
        ],
    },
    "delta_f1": float(np.mean(f1_v3) - np.mean(f1_baseline)),
    "beat_crisisspot": bool(np.mean(f1_v3) > 0.909),
}

results_path = "results/experiment_results.json"
with open(results_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Results saved to: {results_path}")
print(f"\n📋 Summary:")
print(json.dumps(summary, indent=2))

# Copy kết quả lên Google Drive (nếu có)
if IN_COLAB and os.path.exists('/content/drive/MyDrive'):
    drive_dir = '/content/drive/MyDrive/CausalCrisis_Results'
    os.makedirs(drive_dir, exist_ok=True)
    !cp results/experiment_results.json {drive_dir}/
    !cp -f checkpoints/*.pt {drive_dir}/ 2>/dev/null || true
    !cp -f *.png {drive_dir}/ 2>/dev/null || true
    print(f"\n☁️ Results also saved to Google Drive: {drive_dir}")

---
## 🎉 Experiment Complete!

### Next Steps

1. **Nếu V3 > 90%:** Tiến hành ablation study (Cell tiếp theo)
2. **Nếu V3 < 90%:** Phân tích error cases, tune hyperparameters
3. **LODO evaluation:** Test domain generalization
4. **Paper writing:** Sử dụng `ml-paper-writing` skill

In [ ]:
print("\n" + "🎉" * 20)
print("\n  🏁 CausalCrisis V3 Experiment Complete!")
print(f"\n  📊 Final Score: F1 = {np.mean(f1_v3):.4f} ± {np.std(f1_v3):.4f}")
print(f"  📈 vs Baseline: Δ = {np.mean(f1_v3) - np.mean(f1_baseline):+.4f}")
if np.mean(f1_v3) > 0.909:
    print("  🏆 CrisisSpot BEATEN! New SOTA! 🎊")
else:
    gap = 0.909 - np.mean(f1_v3)
    print(f"  📏 Gap to CrisisSpot: {gap:.4f} — keep optimizing!")
print("\n" + "🎉" * 20)